# 00 — Verificación del entorno de desarrollo

**Propósito del notebook:** Verificar que Visual Studio Code, WSL2, Jupyter, el entorno virtual `.venv`, TensorFlow/Keras, la GPU y la arquitectura EfficientNet-B4 estén correctamente configurados antes de implementar el pipeline experimental sobre ISIC 2019.

**Tarea experimental:** Clasificación de las ocho categorías diagnósticas etiquetadas en el conjunto de entrenamiento ISIC 2019: melanoma (`MEL`), nevo melanocítico (`NV`), carcinoma basocelular (`BCC`), queratosis actínica (`AK`), queratosis benigna (`BKL`), dermatofibroma (`DF`), lesión vascular (`VASC`) y carcinoma espinocelular (`SCC`).

**Criterio de reproducibilidad:** Todas las verificaciones se ejecutan con el intérprete definido para el proyecto, ubicado en `/proyecto_de_titulo/.venv/bin/python`. El desarrollo y las pruebas breves se realizan localmente en WSL2; las ejecuciones experimentales completas se ejecutarán en el HPC Océano PUCV.

In [1]:
import platform
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

print(f"Directorio de trabajo : {PROJECT_ROOT}")
print(f"Ejecutable de Python  : {sys.executable}")
print(f"Versión de Python     : {sys.version.split()[0]}")
print(f"Sistema operativo     : {platform.platform()}")

Directorio de trabajo : /home/milat/tesis_melanoma/notebooks
Ejecutable de Python  : /home/milat/tesis_melanoma/.venv/bin/python
Versión de Python     : 3.10.12
Sistema operativo     : Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.35


## Verificación de dependencias

A continuación se comprueba la disponibilidad de las bibliotecas principales para manipulación de datos, visualización, aprendizaje automático, procesamiento de imágenes y entrenamiento de redes neuronales. Esta verificación no instala ni modifica paquetes.

In [2]:
import importlib.util

PAQUETES = {
    "numpy": "Cálculo numérico",
    "pandas": "Manipulación de datos tabulares",
    "matplotlib": "Visualización básica",
    "seaborn": "Visualización estadística",
    "sklearn": "Métricas y utilidades de machine learning",
    "PIL": "Lectura y manipulación de imágenes",
    "cv2": "Visión computacional con OpenCV",
    "torch": "Deep learning con PyTorch",
    "torchvision": "Utilidades y transformaciones para visión",
    "timm": "Modelos de visión preentrenados, incluido EfficientNet-B4",
    "ipykernel": "Kernel de Jupyter",
}

for paquete, descripcion in PAQUETES.items():
    disponible = importlib.util.find_spec(paquete) is not None
    estado = "✓ INSTALADO" if disponible else "✗ NO INSTALADO"
    print(f"{estado:16} | {paquete:12} | {descripcion}")

✓ INSTALADO      | numpy        | Cálculo numérico
✓ INSTALADO      | pandas       | Manipulación de datos tabulares
✓ INSTALADO      | matplotlib   | Visualización básica
✓ INSTALADO      | seaborn      | Visualización estadística
✓ INSTALADO      | sklearn      | Métricas y utilidades de machine learning
✓ INSTALADO      | PIL          | Lectura y manipulación de imágenes
✓ INSTALADO      | cv2          | Visión computacional con OpenCV
✓ INSTALADO      | torch        | Deep learning con PyTorch
✓ INSTALADO      | torchvision  | Utilidades y transformaciones para visión
✓ INSTALADO      | timm         | Modelos de visión preentrenados, incluido EfficientNet-B4
✓ INSTALADO      | ipykernel    | Kernel de Jupyter


In [3]:
from importlib.metadata import PackageNotFoundError, version

DISTRIBUCIONES = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "Pillow",
    "opencv-python",
    "torch",
    "torchvision",
    "timm",
    "ipykernel",
]

for distribucion in DISTRIBUCIONES:
    try:
        print(f"{distribucion:16} {version(distribucion)}")
    except PackageNotFoundError:
        print(f"{distribucion:16} NO INSTALADO")

numpy            1.26.4
pandas           2.3.3
matplotlib       3.10.9
seaborn          0.13.2
scikit-learn     1.7.2
Pillow           12.3.0
opencv-python    5.0.0.93
torch            2.14.0
torchvision      0.29.0
timm             1.0.29
ipykernel        7.3.0


## Verificación de TensorFlow, Keras y aceleración por GPU

La implementación del proyecto utiliza TensorFlow y Keras como framework de aprendizaje profundo. En esta sección se verifica la versión instalada, la interfaz Keras disponible y la detección de dispositivos físicos por TensorFlow.

La disponibilidad de una GPU no determina la validez funcional del código, pero reduce significativamente el tiempo de entrenamiento de EfficientNet-B4. Por ello, este dato se documenta antes de ejecutar experimentos.

In [4]:
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
print(f"Keras     : {tf.keras.__version__}")

dispositivos_gpu = tf.config.list_physical_devices("GPU")
dispositivos_cpu = tf.config.list_physical_devices("CPU")

print(f"\nCPU detectadas: {len(dispositivos_cpu)}")
print(f"GPU detectadas: {len(dispositivos_gpu)}")

if dispositivos_gpu:
    for indice, gpu in enumerate(dispositivos_gpu):
        print(f"GPU {indice}: {gpu.name}")
else:
    print(
        "TensorFlow no detectó una GPU. "
        "Las verificaciones iniciales podrán ejecutarse en CPU, "
        "pero el entrenamiento completo se realizará más eficientemente con GPU."
    )

2026-09-18 17:49:34.081211: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-18 17:49:34.928467: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


TensorFlow: 2.16.1
Keras     : 3.12.4

CPU detectadas: 1
GPU detectadas: 1
GPU 0: /physical_device:GPU:0


2026-09-18 17:49:36.652354: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-18 17:49:36.967194: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-18 17:49:36.967412: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


### Resultado de la verificación

La ejecución confirmó que el entorno cuenta con TensorFlow 2.16.1 y Keras 3.12.4. TensorFlow detectó una CPU y una GPU disponibles en WSL2, por lo que los experimentos con EfficientNet-B4 pueden ejecutarse con aceleración por GPU.

El proyecto se ejecuta en Ubuntu 22.04 mediante Windows Subsystem for Linux 2 (WSL2), aunque Visual Studio Code se utiliza desde Windows. Esta configuración se adoptó porque TensorFlow con versiones posteriores a 2.10 no cuenta con soporte oficial para aceleración CUDA mediante GPU en Windows nativo. WSL2 permite mantener TensorFlow 2.16.1 y utilizar la GPU NVIDIA del equipo desde un entorno Linux compatible.

Los mensajes informativos relacionados con AVX2/FMA corresponden a optimizaciones de CPU; el aviso sobre TensorRT solo indica que dicho componente opcional no está instalado. Los avisos sobre NUMA son habituales en algunos entornos WSL2 y no impiden el uso de la GPU por TensorFlow.

En consecuencia, los notebooks, scripts y experimentos de este proyecto deben ejecutarse desde Visual Studio Code conectado a `WSL: Ubuntu-22.04`, usando el intérprete del entorno virtual ubicado en `/proyecto_de_titulo/.venv/bin/python`.

## Verificación de EfficientNet-B4

EfficientNet-B4 será la arquitectura base del proyecto. La siguiente prueba construye el modelo mediante `tf.keras.applications` sin pesos preentrenados, con el fin de verificar únicamente que la arquitectura es compatible con el entorno configurado.

La entrada se define como una imagen dermatoscópica RGB de 380 × 380 píxeles. En etapas posteriores se utilizará transferencia de aprendizaje, inicialmente con pesos ImageNet, y se reemplazará la capa de clasificación final para adecuarla a las ocho clases diagnósticas de ISIC 2019.

In [1]:
from tensorflow.keras.applications import EfficientNetB4

IMAGE_SIZE = 380
CHANNELS = 3
NUM_CLASSES = 8

modelo_prueba = EfficientNetB4(
    include_top=False,
    weights=None,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS),
    pooling="avg",
)

print("EfficientNet-B4 se construyó correctamente.")
print(f"Forma de entrada: {modelo_prueba.input_shape}")
print(f"Forma de salida del extractor: {modelo_prueba.output_shape}")
print(f"Parámetros totales: {modelo_prueba.count_params():,}")
print(f"Clases objetivo posteriores: {NUM_CLASSES}")

2026-09-18 18:25:18.655407: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-18 18:25:20.277259: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-09-18 18:25:24.405775: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-18 18:25:24.406367: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required li

EfficientNet-B4 se construyó correctamente.
Forma de entrada: (None, 380, 380, 3)
Forma de salida del extractor: (None, 1792)
Parámetros totales: 17,673,823
Clases objetivo posteriores: 8


### Resultado de la verificación de EfficientNet-B4

La arquitectura EfficientNet-B4 se construyó correctamente mediante TensorFlow/Keras, sin cargar pesos preentrenados y sin realizar entrenamiento. El modelo recibió como entrada imágenes RGB de 380 × 380 píxeles y produjo un vector de características de 1792 dimensiones mediante agrupamiento promedio global.

La instancia generada contiene 17.673.823 parámetros. En la etapa experimental se utilizarán pesos preentrenados en ImageNet y se incorporará una nueva cabeza de clasificación para las ocho categorías diagnósticas de ISIC 2019: MEL, NV, BCC, AK, BKL, DF, VASC y SCC.

Durante la ejecución, TensorFlow reconoció la GPU NVIDIA GeForce RTX 2060, con aproximadamente 4 GB de memoria disponible y capacidad de cómputo 7.5. Por tanto, el entorno local es apto para ejecutar verificaciones, pruebas de humo y entrenamiento breve mediante aceleración por GPU. Las corridas experimentales completas se ejecutarán en el HPC Océano PUCV.

Los avisos asociados a NUMA corresponden a la virtualización de hardware propia de WSL2. TensorFlow creó correctamente el dispositivo `GPU:0`; por ello, estos mensajes no representan un error ni requieren una modificación de la configuración.

## Entornos de desarrollo y ejecución experimental

El proyecto utiliza dos entornos de cómputo complementarios. El desarrollo inicial se realiza localmente mediante Visual Studio Code conectado a Ubuntu 22.04 a través de WSL2, utilizando TensorFlow/Keras y la GPU NVIDIA GeForce RTX 2060 disponible en el equipo.

El entrenamiento experimental completo, el ajuste fino de hiperparámetros y las repeticiones necesarias para obtener resultados reproducibles se ejecutarán en el HPC Océano de la Pontificia Universidad Católica de Valparaíso. Esta separación permite utilizar el equipo local para desarrollar, depurar y validar el pipeline, mientras que el clúster se emplea para las ejecuciones de mayor costo computacional.

Para asegurar portabilidad y reproducibilidad, las rutas de datos se definirán de manera relativa o mediante parámetros de configuración. Asimismo, las configuraciones para pruebas locales y ejecuciones en el HPC se mantendrán separadas de la lógica principal del código.